In [ ]:
import os
os.chdir(r'c:\Users\leopa\Documents\python codes\New folder\Information_retrieval\notebooks')
print(f"Current working directory: {os.getcwd()}")

# Step 11: Model G — Triple-RRF Ensemble + Cross-Encoder Re-ranking

## Architecture

This is the most advanced pipeline tested. It follows a **two-stage retrieval-then-reranking** paradigm used in production-grade search systems (Elasticsearch, Pinecone).

```
Stage 1: Retrieval — Triple Reciprocal Rank Fusion
    ├── BM25 (keywords)            → rank list
    ├── TF-IDF + Cosine Sim        → rank list
    └── all-mpnet-base-v2 (dense)  → rank list
         └── RRF combines ranks → Top 50 candidates

Stage 2: Re-ranking — Cross-Encoder
    └── cross-encoder/ms-marco-MiniLM-L-6-v2
    └── Takes (query, candidate) pairs → relevance score
    └── Final ranking by cross-encoder score
```

### Why RRF instead of Score Fusion?
Reciprocal Rank Fusion (Cormack et al., 2009) fuses **ranks**, not scores:
$$RRF\_score(d) = \sum_{r \in rankers} \frac{1}{k + rank_r(d)}$$
This is **distribution-agnostic** — it doesn't matter that BM25 scores are in [0, 300] and cosine scores in [0, 1]. Only ordering matters. This avoids the normalization fragility of Model D/F.

### Why Cross-Encoder?
A **bi-encoder** (previous models) encodes query and document independently. A **cross-encoder** takes both as a single input, enabling full attention between every token of both — producing far more accurate relevance scores, at the cost of speed (hence, only top-50 candidates).

In [ ]:
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split (Same Seed as All Other Models)

In [ ]:
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places  = pd.read_csv('../data/Tripadvisor.csv', low_memory=False)

eval_cols  = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places  = df_places[eval_cols].copy()

df_merged  = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')
print(f"Total valid places: {len(df_merged)}")

np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy()
test_df  = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train (Queries): {len(train_df)} | Test (Corpus): {len(test_df)}")

### 2. Evaluation Functions (Type-Aware)

In [ ]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    """Level 1 error: rank of first match on typeR. Error = rank index."""
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    Type-aware subcategory extraction.
    Per assignment spec:
      H  (Hotel)          -> priceRange
      R  (Restaurant)     -> restaurantType, restaurantTypeCuisine
      A / AP (Attraction) -> activiteSubType
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                for p in str(val).split(','):
                    cats.add(p.strip().lower())
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    return cats

def eval_level_2(query_subcats, sorted_test_indices, test_subcats_list):
    """Level 2 error: rank of first test place sharing at least one subcategory."""
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]
test_df['subcategories'] = test_subcats_list
print("Evaluation functions ready.")

### 3. Stage 1a — BM25 Index

In [ ]:
print("[Stage 1a] Building BM25 index...")
corpus_kw = test_df['top_100_words'].fillna('').astype(str).tolist()
bm25 = BM25Okapi([doc.split(' ') for doc in corpus_kw])
print("BM25 indexed.")

### 4. Stage 1b — TF-IDF Cosine Similarity

In [ ]:
print("[Stage 1b] Building TF-IDF vectors...")
train_kw = train_df['top_100_words'].fillna('').tolist()
test_kw  = test_df['top_100_words'].fillna('').tolist()

tfidf_vectorizer = TfidfVectorizer()
tfidf_train = tfidf_vectorizer.fit_transform(train_kw)
tfidf_test  = tfidf_vectorizer.transform(test_kw)
tfidf_sims  = cosine_similarity(tfidf_train, tfidf_test)
print(f"TF-IDF matrix: {tfidf_sims.shape}")

### 5. Stage 1c — Dense Transformer (all-mpnet-base-v2)

Upgraded from `all-MiniLM-L6-v2` (6 layers, 384-dim) to `all-mpnet-base-v2` (12 layers, 768-dim) for significantly improved semantic understanding.

> ⚠️ **Note:** This encoding step takes ~5–10 minutes on CPU.

In [ ]:
print("[Stage 1c] Encoding with all-mpnet-base-v2 (768-dim, 12 layers)...")
bi_encoder = SentenceTransformer('all-mpnet-base-v2')

train_vecs = bi_encoder.encode(train_kw, show_progress_bar=True, batch_size=64)
test_vecs  = bi_encoder.encode(test_kw,  show_progress_bar=True, batch_size=64)
dense_sims = cosine_similarity(train_vecs, test_vecs)
print(f"Dense similarity matrix: {dense_sims.shape}")

### 6. Stage 2 — Cross-Encoder Setup

In [ ]:
print("[Stage 2] Loading Cross-Encoder...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("Cross-Encoder ready.")

### 7. Reciprocal Rank Fusion Helper

In [ ]:
def reciprocal_rank_fusion(*rank_lists, k=60):
    """
    Reciprocal Rank Fusion (Cormack et al., 2009).

    Given N rank lists (each a list of document indices sorted by relevance),
    compute for each document:
        RRF_score(d) = sum_{r in rankers} 1 / (k + rank_r(d))

    k=60 is the standard smoothing constant from the original paper.
    Returns indices sorted by descending RRF score.
    """
    rrf_scores = {}
    for rank_list in rank_lists:
        for rank, doc_idx in enumerate(rank_list):
            rrf_scores.setdefault(doc_idx, 0.0)
            rrf_scores[doc_idx] += 1.0 / (k + rank)
    return sorted(rrf_scores.keys(), key=lambda d: rrf_scores[d], reverse=True)

print("RRF function defined.")

### 8. Full Pipeline Evaluation

> ⚠️ **Note:** This loop takes ~30 minutes on CPU due to the Cross-Encoder inference per query.

In [ ]:
TOP_K = 50  # Number of RRF candidates to re-rank with the Cross-Encoder

lvl1_errors, lvl2_errors = [], []
start_time = time.time()
n_queries  = len(train_df)
print(f"Evaluating {n_queries} queries (RRF + Cross-Encoder re-rank top {TOP_K})...")

for i in range(n_queries):
    row = train_df.iloc[i]
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        continue

    # --- Rank lists from each retriever ---
    bm25_ranked  = np.argsort(bm25.get_scores(query_text.split(' ')))[::-1].tolist()
    tfidf_ranked = np.argsort(tfidf_sims[i])[::-1].tolist()
    dense_ranked = np.argsort(dense_sims[i])[::-1].tolist()

    # --- Reciprocal Rank Fusion ---
    rrf_ranked = reciprocal_rank_fusion(bm25_ranked, tfidf_ranked, dense_ranked, k=60)

    # --- Cross-Encoder Re-ranking on Top-K ---
    top_k_idx = rrf_ranked[:TOP_K]
    pairs     = [(query_text, test_kw[idx]) for idx in top_k_idx]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)

    reranked_top = [top_k_idx[j] for j in np.argsort(ce_scores)[::-1]]
    full_ranked  = reranked_top + rrf_ranked[TOP_K:]

    # --- Evaluate ---
    test_typeR_list = test_df['typeR'].values[full_ranked].tolist()

    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None: lvl1_errors.append(err_1)

    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, full_ranked, test_subcats_list)
    if err_2 is not None: lvl2_errors.append(err_2)

    if (i + 1) % 100 == 0:
        print(f"  ... processed {i+1}/{n_queries} queries")

elapsed = time.time() - start_time
print(f"\nEvaluation done in {elapsed:.2f}s")
print("-" * 40)
print(f"Model G (Triple-RRF + Cross-Encoder) Average Ranking Error Level 1: {np.mean(lvl1_errors):.2f}")
print(f"Model G (Triple-RRF + Cross-Encoder) Average Ranking Error Level 2: {np.mean(lvl2_errors):.2f}")
print(f"  (Computed on {len(lvl1_errors)} L1 queries, {len(lvl2_errors)} L2 queries)")

---
## Discussion

| Model | L1 Error | L2 Error |
|:------|:--------:|:--------:|
| BM25 Baseline | 0.64 | 4.41 |
| Model F (Best Hybrid 85/15) | 0.67 | **4.19** |
| **Model G (Triple-RRF + Cross-Encoder)** | 0.71 | 4.65 |

**Why did the most complex model lose to the simpler one?**

This is a classic **over-engineering** result. The RRF + Cross-Encoder pipeline was designed for long-form natural language text (e.g., legal documents, news articles), where semantic similarity is harder to measure. In our case, the input is already **TF-IDF keyword signatures** — highly concentrated, discriminative tokens with minimal noise.

The Cross-Encoder introduces "hallucinated" semantic similarities: it sees (query keywords, candidate keywords) and tries to model relevance as a passage-retrieval task. The MS-MARCO training objective doesn't perfectly match our venue-similarity task, and the model sometimes boosts semantically "interesting" but categorically wrong venues to the top.

**Conclusion:** For keyword-signature based retrieval on this dataset, a **Weighted Score Fusion (Model F)** remains the strongest approach.